In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["OPENCV_LOG_LEVEL"] = "ERROR"

In [5]:
from pathlib import Path

import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm

from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [6]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]


## Preprocess Datasets


In [5]:
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.models.wrappers.automorph import automorph_segment_av
from fundus_vessels_toolkit.models.wrappers.vascx import vascx_segment_av

for dataset_path in DATASETS_PATH.values():
    raw_path = dataset_path / "1-images"
    for raw_file in tqdm(Path(raw_path).glob(f"*{most_common_image_ext(raw_path)}")):
        fvt_out = dataset_path / "2-av-pred_FVT" / (raw_file.stem + ".png")
        automorph_out = dataset_path / "2-av-pred_Automorph" / (raw_file.stem + ".png")
        vascx_out = dataset_path / "2-av-pred_VascX" / (raw_file.stem + ".png")

        if fvt_out.exists() and automorph_out.exists() and vascx_out.exists():
            continue

        fundus = FundusData(image=raw_file)
        fundus_cropped, roi = fundus.crop_to_roi(return_roi=True)

        if not fvt_out.exists():
            segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=fvt_out)

        if not automorph_out.exists():
            automorph_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=automorph_out)

        if not vascx_out.exists():
            vascx_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=vascx_out)

50it [00:00, 25628.16it/s]
200it [00:00, 26203.75it/s]
100it [00:00, 30450.88it/s]
44it [00:00, 25578.57it/s]
22it [00:00, 27044.16it/s]
15it [00:00, 19853.13it/s]
20it [00:00, 21394.05it/s]
20it [00:00, 29495.81it/s]


In [ ]:
dataset = BranchDigraphDataset.load_from_dirs(
    RAW,
    TOPO,
    AV,
    dataset_name=list(DATASETS_PATH.keys()),
    resize_to=1024,
    output_dir="tmp/ALL_DATA_V2",
    n_workers=0,
    mask_optic_disc=True,
)

Found 373 branch digraphs...


Processing...
Done!


In [13]:
augment = dataset.cfg.augment.model_copy(update={"elastic": None, "rotate": None, "horizontal_flip": False})
dataset.jppype_show(1, augment=augment)[0]

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:438: UserWarning: Missing attribute 'CALIBRES' for branch 102.
  super().__init__(


GridBox(children=(HTML(value='<h3 style="text-align: center;">002_N/automorph</h3>'), HTML(value='<h3 style="t…

## Define dataset splits


In [ ]:
samples_by_dataset: dict[str, list[SampleInfo]] = {}
for sample in dataset.samples_info:
    samples_by_dataset.setdefault(sample.dataset, []).append(sample)

samples_stratification: dict[str, list[SampleInfo]] = {}


**Fundus AV**: stratify by pathology


In [ ]:
for sample in samples_by_dataset["FundusAV"]:
    samples_stratification.setdefault(f"FundusAV-{sample.name[-1]}", []).append(sample)

**LES-AV**, **INSPIRE** and **GAVE-train**: no specific stratification


In [ ]:
samples_stratification["LES-AV"] = samples_by_dataset["LES-AV"]
samples_stratification["INSPIRE"] = samples_by_dataset["INSPIRE"]
samples_stratification["GAVE-train"] = samples_by_dataset["GAVE-train"]
samples_stratification["HRF"] = samples_by_dataset["HRF"]

In [ ]:
for samples in samples_stratification.values():
    N = len(samples)
    train_end, val_end = int(N * 0.7), int(N * 0.85)
    for sample in samples[:train_end]:
        sample.dataset_type = "train"
    for sample in samples[train_end:val_end]:
        sample.dataset_type = "validation"
    for sample in samples[val_end:]:
        sample.dataset_type = "test"

Use existing splits for **DRIVE** and **MAPLES-DR**


In [ ]:
for sample in samples_by_dataset["DRIVE_train"]:
    sample.dataset_type = "validation" if sample.name.startswith(("31", "33", "35", "28")) else "train"
for sample in samples_by_dataset["DRIVE_test"]:
    sample.dataset_type = "test"

In [ ]:
import maples_dr

maples_dr_test_samples_name = {s.name for s in maples_dr.load_test_set()}
maples_dr_test_samples: list[SampleInfo] = []
for sample in samples_by_dataset["MAPLES-DR"]:
    if sample.name in maples_dr_test_samples_name:
        maples_dr_test_samples.append(sample)
    else:
        sample.dataset_type = "train"
N_test = len(maples_dr_test_samples)
for sample in maples_dr_test_samples[: N_test // 2]:
    sample.dataset_type = "validation"
for sample in maples_dr_test_samples[N_test // 2 :]:
    sample.dataset_type = "test"

Check that all samples have a dataset type assigned and save the splits


In [ ]:
samples_without_type = [
    sample for sample in dataset.samples_info if sample.dataset_type not in ("train", "validation", "test")
]
assert not samples_without_type, f"Samples without dataset type: {[s.name for s in samples_without_type]}"

dataset.save_manifest()

In [ ]:
train_set, val_set, test_set = dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

## Bundle dataset


In [ ]:
dataset.bundle("ALL_DATA_bundle.tar.gz", overwrite=True)

In [ ]:
bundled_dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

In [ ]:
train_set, val_set, test_set = bundled_dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

## Test


In [7]:
samples_src = BranchDigraphDataset.discover_paths(RAW, TOPO, AV, dataset_name=list(DATASETS_PATH.keys()))

In [21]:
from fundus_vessels_toolkit.utils.profiling import Profiler

with Profiler():
    sample_info = samples_src[0].process(
        output_dir=Path("tmp/test_data_process/"), resize_to=1024, mask_optic_disc=True, overwrite=True
    )
    sample = sample_info.load(load_av_maps=True)

2


]8;id=739362;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:441\SampleSource.process]8;;\                    2.8s                    (runs=1)
├── ]8;id=34561;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:443\Load fundus image]8;;\                  ↳3.6%  101.6ms           (runs=1)
│   ├── ]8;id=416951;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:444\Read image from disk]8;;\           0.71%    ↳ 20%   20.0ms  (runs=1)
│   ├── ]8;id=261735;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:447\Crop to ROI & resize]8;;\           2.65%    ↳ 73%   74.6ms  (runs=1)
│   └── ]8;id=681588;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:463\Write processed fundus image]8;;\   0.24%    ↳6.6%    6.7ms  (runs=1)
├── ]8;id=429614;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:468\Load or compute OD and Macula]8;;\      ↳4.4%  124.8ms           (runs=1)
│   └── ]8;id=9415;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:382\Segment OD and Macula]8;;\          2.85%    ↳ 64%   80.2ms  (runs=1)
├── ]8;id=263483;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:481\Load and preprocess graphes]8;;\        ↳ 70%     2.0s           (runs=1)
│   ├── ]8;id=931200;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:520\Load AV map]8;;\                      10%    ↳ 14%  282.0ms  (runs=4, avg=  70.5ms)
│   ├── ]8;id=791646;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:529\av2tree.to_vgraph]8;;\                39%    ↳ 56%     1.1s  (runs=4, avg= 274.8ms)
│   ├── ]8;id=353481;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:533\Clean graph]8;;\                    1.47%    ↳2.1%   41.3ms  (runs=4, avg=  10.3ms)
│   ├── ]8;id=25507;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:540\Check graph]8;;\                    0.84‰    ↳0.1%    2.4ms  (runs=4, avg= 594.8µs)
│   └── ]8;id=583560;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:547\Save processed graphes]8;;\         0.39%    ↳0.5%   10.8ms  (runs=4, avg=   2.7ms)
└── ]8;id=127647;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:557\Load and preprocess GT topology]8;;\    ↳ 22%  613.5ms           (runs=1)
    ├── ]8;id=329538;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:561\VTree.load]8;;\                     0.64%    ↳2.9%   18.1ms  (runs=2, avg=   9.0ms)
    ├── ]8;id=63743;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:566\Transform VTree]8;;\                1.04%    ↳4.8%   29.2ms  (runs=2, avg=  14.6ms)
    ├── ]8;id=748273;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:570\Clean]8;;\                          0.33%    ↳1.5%    9.2ms  (runs=2, avg=   4.6ms)
    ├── ]8;id=356473;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:581\Check common branches]8;;\          0.77%    ↳3.5%   21.7ms  (runs=1)
    ├── ]8;id=665258;file:///home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/models/topology/dataset.py:591\Rasterize topologies]8;;\             13%    ↳ 62%  378.9ms  (runs=1)
    └── ]8;id=963160;file:///home/gaby/Lab/Src/fundus-

In [22]:
sample.show(graph_version="fvt")

View2D()